## Startup cell for every new run

In [ ]:
import os
import sys
import shutil
import time
import zipfile
import json
import subprocess
import torch
import numpy as np
from google.colab import drive
from functools import partialmethod
import yaml
import tqdm

OUTPUT_PATH = "/content/drive/MyDrive/EE243 Project/outputs"

In [ ]:
# ── Cell 1: Full Setup ──
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/spla-tam/SplaTAM.git /content/SplaTAM
!pip install -q tqdm==4.65.0 kornia lpips open3d wandb pytorch-msssim natsort imageio torchmetrics plyfile==0.8.1
!pip install -q git+https://github.com/JonathonLuiten/diff-gaussian-rasterization-w-depth.git
!pip uninstall -q datasets -y  # prevents HuggingFace datasets shadowing SplaTAM's datasets/

# Fix missing __init__.py
!touch /content/SplaTAM/datasets/gradslam_datasets/__init__.py

# Patch configs
!sed -i 's/scene_name = scenes\[0\]/scene_name = scenes[3]/' /content/SplaTAM/configs/replica/splatam.py
!sed -i 's|basedir="./data/Replica"|basedir="/content/data/Replica"|' /content/SplaTAM/configs/replica/splatam.py
!sed -i 's|workdir="./experiments/Replica"|workdir="/content/drive/MyDrive/EE243 Project/outputs"|' /content/SplaTAM/configs/replica/splatam.py
!sed -i 's/use_wandb=True/use_wandb=False/' /content/SplaTAM/configs/replica/splatam.py
!sed -i 's/save_checkpoints=False/save_checkpoints=True/' /content/SplaTAM/configs/replica/splatam.py

print("Setup complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into '/content/SplaTAM'...
remote: Enumerating objects: 210, done.
remote: Counting objects: 100% (168/168), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 210 (delta 109), reused 106 (delta 106), pack-reused 42 (from 1)
Receiving objects: 100% (210/210), 14.34 MiB | 45.61 MiB/s, done.
Resolving deltas: 100% (111/111), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Verifying the cloning of repo into colab runtime

In [ ]:
# Cell 1a — verify SplaTAM repo is intact
splatam_files = ["scripts/splatam.py", "configs/replica/splatam.py", "utils/common_utils.py"]
for f in splatam_files:
    path = f"/content/SplaTAM/{f}"
    print(f"{'OK' if os.path.exists(path) else 'MISSING'}: {f}")

OK: scripts/splatam.py
OK: configs/replica/splatam.py
OK: utils/common_utils.py


In [ ]:
# Cell 1b — verify the CUDA rasterizer installed
try:
    from diff_gaussian_rasterization import GaussianRasterizer
    print("OK: diff_gaussian_rasterization imported")
except ImportError as e:
    print(f"FAILED: {e}")
    print("Re-run: !pip install git+https://github.com/JonathonLuiten/diff-gaussian-rasterization-w-depth.git")

OK: diff_gaussian_rasterization imported


In [ ]:
# Cell 1c — GPU + torch sanity check
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")
print(f"Free VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB" if torch.cuda.is_available() else "")

Torch: 2.11.0+cu128
CUDA available: True
Device: Tesla T4
Free VRAM: 15.53 GB


## Unzip Replica into runtime (only for Replica baseline testing)

Make sure the Replica.zip is sitting in your data/ directory before running this.

In [ ]:
# Unzip data
os.makedirs("/content/data", exist_ok=True)
!unzip -q "/content/drive/MyDrive/EE243 Project/data/Replica.zip" -d "/content/data/"

In [ ]:
ZIP_PATH = "/content/drive/MyDrive/EE243 Project/data/Replica.zip"
LOCAL_DATA = "/content/data"  # fast local disk

os.makedirs(LOCAL_DATA, exist_ok=True)

# Check free local disk first
total, used, free = shutil.disk_usage("/content")
print(f"Local disk: {free/1e9:.1f} GB free of {total/1e9:.1f} GB total")
print(f"Zip size: {os.path.getsize(ZIP_PATH)/1e9:.1f} GB (extracted will be ~2-3x)\n")

start = time.time()
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    members = z.namelist()
    total_files = len(members)
    print(f"Extracting {total_files} files to {LOCAL_DATA}...")
    for i, name in enumerate(members):
        z.extract(name, LOCAL_DATA)
        if i % 1000 == 0 and i > 0:
            elapsed = time.time() - start
            rate = i / elapsed
            eta = (total_files - i) / rate
            print(f"  {i}/{total_files} — {elapsed:.0f}s elapsed, ~{eta:.0f}s remaining")

print(f"\nDone in {(time.time()-start)/60:.1f} min")

Local disk: 178.9 GB free of 253.1 GB total
Zip size: 12.4 GB (extracted will be ~2-3x)

Extracting 32035 files to /content/data...
  1000/32035 — 4s elapsed, ~112s remaining
  2000/32035 — 6s elapsed, ~88s remaining
  3000/32035 — 8s elapsed, ~78s remaining
  4000/32035 — 10s elapsed, ~72s remaining
  5000/32035 — 13s elapsed, ~68s remaining
  6000/32035 — 15s elapsed, ~64s remaining
  7000/32035 — 17s elapsed, ~62s remaining
  8000/32035 — 20s elapsed, ~59s remaining
  9000/32035 — 22s elapsed, ~56s remaining
  10000/32035 — 25s elapsed, ~54s remaining
  11000/32035 — 27s elapsed, ~51s remaining
  12000/32035 — 29s elapsed, ~49s remaining
  13000/32035 — 31s elapsed, ~46s remaining
  14000/32035 — 33s elapsed, ~43s remaining
  15000/32035 — 35s elapsed, ~40s remaining
  16000/32035 — 38s elapsed, ~38s remaining
  17000/32035 — 40s elapsed, ~36s remaining
  18000/32035 — 42s elapsed, ~33s remaining
  19000/32035 — 45s elapsed, ~31s remaining
  20000/32035 — 47s elapsed, ~28s remaining

In [ ]:
DATA_PATH = "/content/data/Replica"  # or whatever the zip extracts to — check structure after

# Verify:
for item in sorted(os.listdir(DATA_PATH)):
    print(item)

.ipynb_checkpoints
cam_params.json
office0
office0_mesh.ply
office1
office1_mesh.ply
office2
office2_mesh.ply
office3
office3_mesh.ply
office4
office4_mesh.ply
room0
room0_mesh.ply
room1
room1_mesh.ply
room2
room2_mesh.ply


In [ ]:
DATA_PATH = "/content/data/Replica/"
expected = ["room0", "office0", "office3"]

for seq in expected:
    path = os.path.join(DATA_PATH, seq)
    exists = os.path.isdir(path)
    if exists:
        results_dir = os.path.join(path, "results")
        n_frames = len(os.listdir(results_dir)) if os.path.isdir(results_dir) else 0
        print(f"{seq}: OK — {n_frames} files in results/")
    else:
        print(f"{seq}: MISSING at {path}")

room0: OK — 4000 files in results/
office0: OK — 4000 files in results/
office3: OK — 4000 files in results/


#Run splatam on room0

In [ ]:
# update the data path in configs
sys.path.insert(0, '/content/SplaTAM')

seq = "room0"
seq_path = os.path.join(DATA_PATH, seq)

cfg_src = "/content/SplaTAM/configs/replica/splatam.py"
cfg_dst = "/content/project/configs/replica_room0.py"
shutil.copy(cfg_src, cfg_dst)

with open(cfg_dst) as f:
    cfg = f.read()

cfg = cfg.replace(
    "input_folder = 'data/Replica/room0'",
    f"input_folder = '{DATA_PATH}/room0'"
).replace(
    "output = 'experiments/Replica/room0'",
    f"output = '{OUTPUT_PATH}/room0'"
)

with open(cfg_dst, "w") as f:
    f.write(cfg)
print("Config written to", cfg_dst)

Config written to /content/project/configs/replica_room0.py


In [ ]:
init_file = "/content/SplaTAM/datasets/gradslam_datasets/__init__.py"

with open(init_file) as f:
    content = f.read()

# Add geometryutils import if it's missing
if "geometryutils" not in content:
    content = "from . import geometryutils\n" + content
    with open(init_file, "w") as f:
        f.write(content)
    print("Added geometryutils import to __init__.py")
else:
    print("geometryutils already imported")

# Verify
with open(init_file) as f:
    print("\nUpdated __init__.py:")
    print(f.read()[:200])

Added geometryutils import to __init__.py

Updated __init__.py:
from . import geometryutils
from .azure import AzureKinectDataset
from .basedataset import GradSLAMDataset
from .dataconfig import load_dataset_config
from .datautils import *
from .icl import ICLData


In [ ]:
sys.path.insert(0, "/content/SplaTAM")
os.chdir("/content/SplaTAM")

# Silence tqdm bars
tqdm.tqdm.__init__ = partialmethod(tqdm.tqdm.__init__, disable=True)

# Clear cached modules
for key in list(sys.modules.keys()):
    if any(x in key for x in ['datasets', 'gradslam', 'utils']):
        del sys.modules[key]

sys.argv = ["splatam.py", "/content/project/configs/replica_room0.py"]
script_globals = {
    '__file__': '/content/SplaTAM/scripts/splatam.py',
    '__name__': '__main__',
}

# run splatam
print("Launching SplaTAM...", flush=True)
with open("/content/SplaTAM/scripts/splatam.py") as f:
    exec(f.read(), script_globals)

Launching SplaTAM...
System Paths:
/content/SplaTAM
/content/SplaTAM
/content/SplaTAM
/content/SplaTAM
/content/SplaTAM
/content/SplaTAM
/content/SplaTAM
/content/SplaTAM
/content/SplaTAM
/content/SplaTAM
/content/SplaTAM
/content/SplaTAM
/content/SplaTAM
/content/SplaTAM
/content
/env/python
/usr/lib/python312.zip
/usr/lib/python3.12
/usr/lib/python3.12/lib-dynload

/usr/local/lib/python3.12/dist-packages
/usr/lib/python3/dist-packages
/usr/local/lib/python3.12/dist-packages/IPython/extensions
/root/.ipython
Seed set to: 0 (type: <class 'int'>)
Loaded Config:
{'workdir': '/content/drive/MyDrive/EE243 Project/outputs/Replica', 'run_name': 'room0_0', 'seed': 0, 'primary_device': 'cuda:0', 'map_every': 1, 'keyframe_every': 5, 'mapping_window_size': 24, 'report_global_progress_every': 500, 'eval_every': 5, 'scene_radius_depth_ratio': 3, 'mean_sq_dist_method': 'projective', 'gaussian_distribution': 'isotropic', 'report_iter_progress': False, 'load_checkpoint': False, 'checkpoint_time_idx':



Tracking Time Step: 29:  88%|████████▊ | 35/40 [00:18<00:00, 32.79it/s]


Selected Keyframes at Frame 3: [0, 3]

Selected Keyframes at Frame 4: [0, 4]

Selected Keyframes at Frame 5: [0, 4, 5]

Selected Keyframes at Frame 6: [0, 4, 6]

Selected Keyframes at Frame 7: [0, 4, 7]

Selected Keyframes at Frame 8: [0, 4, 8]

Selected Keyframes at Frame 9: [0, 4, 9]

Selected Keyframes at Frame 10: [4, 0, 9, 10]

Selected Keyframes at Frame 11: [4, 0, 9, 11]

Selected Keyframes at Frame 12: [4, 0, 9, 12]

Selected Keyframes at Frame 13: [4, 0, 9, 13]

Selected Keyframes at Frame 14: [0, 4, 9, 14]

Selected Keyframes at Frame 15: [9, 4, 0, 14, 15]

Selected Keyframes at Frame 16: [0, 4, 9, 14, 16]

Selected Keyframes at Frame 17: [9, 0, 4, 14, 17]

Selected Keyframes at Frame 18: [0, 9, 4, 14, 18]

Selected Keyframes at Frame 19: [9, 0, 4, 14, 19]

Selected Keyframes at Frame 20: [14, 9, 4, 0, 19, 20]

Selected Keyframes at Frame 21: [0, 4, 9, 14, 19, 21]

Selected Keyframes at Frame 22: [0, 4, 9, 14, 19, 22]

Selected Keyframes at Frame 23: [0, 4, 14, 9, 19, 23]

S

Mapping Time Step: 61:  70%|███████   | 42/60 [3:51:14<1:39:06, 330.34s/it]


Final Average ATE RMSE: 0.28 cm
Average PSNR: 32.82
Average Depth RMSE: 0.49 cm
Average Depth L1: 0.49 cm
Average MS-SSIM: 0.977
Average LPIPS: 0.072
Saving parameters to: /content/drive/MyDrive/EE243 Project/outputs/Replica/room0_0


In [ ]:
# room 0 run metrics

metrics = {}
for metric in ["psnr", "ssim", "lpips", "rmse", "l1"]:
    with open(f"/content/drive/MyDrive/EE243 Project/outputs/Replica/room0_0/eval/{metric}.txt") as f:
        values = [float(x) for x in f.read().strip().split('\n')]
    metrics[metric] = np.mean(values)
    print(f"{metric.upper()}: {metrics[metric]:.4f}")

PSNR: 32.8211
SSIM: 0.9770
LPIPS: 0.0719
RMSE: 0.0049
L1: 0.0049


In [ ]:
# save metrics
os.makedirs(OUTPUT_PATH + "/baselines", exist_ok=True)
with open(OUTPUT_PATH + "/baselines/replica_room0_results.json", "w") as f:
    json.dump({
        "scene": "replica/room0",
        "paper_psnr": 33.18,
        "our_psnr": 32.82,
        "paper_ssim": 0.972,
        "our_ssim": 0.977,
        "paper_lpips": 0.074,
        "our_lpips": 0.072,
        "notes": "Baseline reproduction on Replica room0"
    }, f, indent=2)
print("Saved")

Saved


# Run Splatam on office0

In [ ]:
# update configs to data path
config_path = "/content/SplaTAM/configs/replica/splatam.py"
with open(config_path) as f:
    config = f.read()

config = config.replace(
    'scene_name = scenes[0]',
    'scene_name = scenes[3]'  # office0 is index 3
).replace(
    'workdir=f"./experiments/{group_name}"',
    f'workdir="/content/drive/MyDrive/EE243 Project/outputs"'
)

with open(config_path, "w") as f:
    f.write(config)
print("Config updated to office0")

Config updated to office0


We will be running the SPLATAM algorithm as a subprocess, and documenting the run log.

In [ ]:
proc = subprocess.Popen(
    ["python", "-u", "scripts/splatam.py", "configs/replica/splatam.py"],
    cwd="/content/SplaTAM",
    stdout=open("/content/drive/MyDrive/EE243 Project/outputs/office0_run.log", "w"),
    stderr=subprocess.STDOUT
)
print(f"Launched PID {proc.pid} — output going to Drive log file")

Launched PID 16120 — output going to Drive log file


In [ ]:
!tail -5 "/content/drive/MyDrive/EE243 Project/outputs/office0_run.log"

Average Depth RMSE: 0.37 cm
Average Depth L1: 0.37 cm
Average MS-SSIM: 0.984
Average LPIPS: 0.082
Saving parameters to: /content/drive/MyDrive/EE243 Project/outputs/office0_0


#Testing on handheld room footage

In [ ]:
!python "/content/drive/MyDrive/EE243 Project/mcap2nerfcapturedataset_results/mcap2dataset.py" \
    --mcap "/content/drive/MyDrive/EE243 Project/data/seq1_room_handheld/seq1_room_handheld_0.mcap" \
    --out  "/content/drive/MyDrive/EE243 Project/mcap2nerfcapturedataset_results/seq1" \
    --rgb-topic   /camera/camera/color/image_raw \
    --depth-topic /camera/camera/aligned_depth_to_color/image_raw \
    --rgb-info    /camera/camera/color/camera_info \
    --depth-info  /camera/camera/color/camera_info

[pass 1] reading CameraInfo...
  color: 640x480 K=
[[381.64291382   0.         316.66802979]
 [  0.         381.02780151 247.90286255]
 [  0.           0.           1.        ]]
  D=[-5.62762581e-02  6.41128719e-02 -6.83844046e-05 -9.62164253e-04
 -2.00389363e-02]
  depth: 640x480 K=
[[381.64291382   0.         316.66802979]
 [  0.         381.02780151 247.90286255]
 [  0.           0.           1.        ]]
  D=[-5.62762581e-02  6.41128719e-02 -6.83844046e-05 -9.62164253e-04
 -2.00389363e-02]
  undistorted color K=
[[377.67479174   0.         315.78856982]
 [  0.         377.47012353 247.77455709]
 [  0.           0.           1.        ]]
[pass 2] loading images...
  loaded 883 rgb / 873 depth messages
[pass 3] syncing + saving pairs...
  saved frame 0 (depth valid 73.0%)
  saved frame 50 (depth valid 75.7%)
  saved frame 100 (depth valid 81.4%)
  saved frame 150 (depth valid 88.3%)
  saved frame 200 (depth valid 78.7%)
  saved frame 250 (depth valid 73.6%)
  saved frame 300 (depth v

In [ ]:
!python "/content/drive/MyDrive/EE243 Project/mcap2nerfcapturedataset_results/mcap2dataset.py" \
    --mcap "/content/drive/MyDrive/EE243 Project/data/seq2_desk_handheld/seq2_desk_handheld_0.mcap" \
    --out  "/content/drive/MyDrive/EE243 Project/mcap2nerfcapturedataset_results/seq2" \
    --rgb-topic   /camera/camera/color/image_raw \
    --depth-topic /camera/camera/aligned_depth_to_color/image_raw \
    --rgb-info    /camera/camera/color/camera_info \
    --depth-info  /camera/camera/color/camera_info

[pass 1] reading CameraInfo...
  color: 640x480 K=
[[381.64291382   0.         316.66802979]
 [  0.         381.02780151 247.90286255]
 [  0.           0.           1.        ]]
  D=[-5.62762581e-02  6.41128719e-02 -6.83844046e-05 -9.62164253e-04
 -2.00389363e-02]
  depth: 640x480 K=
[[381.64291382   0.         316.66802979]
 [  0.         381.02780151 247.90286255]
 [  0.           0.           1.        ]]
  D=[-5.62762581e-02  6.41128719e-02 -6.83844046e-05 -9.62164253e-04
 -2.00389363e-02]
  undistorted color K=
[[377.67479174   0.         315.78856982]
 [  0.         377.47012353 247.77455709]
 [  0.           0.           1.        ]]
[pass 2] loading images...
  loaded 879 rgb / 876 depth messages
[pass 3] syncing + saving pairs...
  saved frame 0 (depth valid 77.7%)
  saved frame 50 (depth valid 75.7%)
  saved frame 100 (depth valid 81.3%)
  saved frame 150 (depth valid 79.5%)
  saved frame 200 (depth valid 70.9%)
  saved frame 250 (depth valid 79.5%)
  saved frame 300 (depth v

In [ ]:
# copying data to colab runtime
os.makedirs("/content/data", exist_ok=True)
print("Copying seq1...")
shutil.copytree(
    "/content/drive/MyDrive/EE243 Project/mcap2nerfcapturedataset_results/seq1",
    "/content/data/seq1"
)
print("Copying seq2...")
shutil.copytree(
    "/content/drive/MyDrive/EE243 Project/mcap2nerfcapturedataset_results/seq2",
    "/content/data/seq2"
)
print("Done.")

Copying seq1...
Copying seq2...
Done.


In [ ]:
os.makedirs("/content/drive/MyDrive/EE243 Project/outputs/turtlebot_seq1", exist_ok=True)

proc = subprocess.Popen(
    ["python", "-u", "scripts/splatam.py", "configs/turtlebot_seq1.py"],
    cwd="/content/SplaTAM",
    stdout=open("/content/drive/MyDrive/EE243 Project/outputs/turtlebot_seq1/run.log", "w"),
    stderr=subprocess.STDOUT
)
print(f"Launched PID {proc.pid}")

Launched PID 7460


In [ ]:
!tail -20 "/content/drive/MyDrive/EE243 Project/outputs/turtlebot_seq1/run.log"

  6%|▌         | 16/273 [00:47<12:49,  3.00s/it]
Traceback (most recent call last):
  File "/content/SplaTAM/scripts/splatam.py", line 1014, in <module>
    rgbd_slam(experiment.config)
  File "/content/SplaTAM/scripts/splatam.py", line 702, in rgbd_slam
    loss.backward()
  File "/usr/local/lib/python3.12/dist-packages/torch/_tensor.py", line 630, in backward
    torch.autograd.backward(
  File "/usr/local/lib/python3.12/dist-packages/torch/autograd/__init__.py", line 364, in backward
    _engine_run_backward(
  File "/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py", line 865, in _engine_run_backward
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
Tracking Time Step: 616:  75%|███████▌  | 30/40 [00:01<00:00, 25.53it/s]


#Testing on handheld desk footage

In [ ]:
os.makedirs("/content/drive/MyDrive/EE243 Project/outputs/turtlebot_seq2", exist_ok=True)
proc = subprocess.Popen(
    ["python", "-u", "scripts/splatam.py", "configs/turtlebot_seq2.py"],
    cwd="/content/SplaTAM",
    stdout=open("/content/drive/MyDrive/EE243 Project/outputs/turtlebot_seq2/run.log", "w"),
    stderr=subprocess.STDOUT
)
print(f"Launched PID {proc.pid}")

Launched PID 9810


In [ ]:
!tail -20 "/content/drive/MyDrive/EE243 Project/outputs/turtlebot_seq2/run.log"

  5%|▍         | 17/376 [00:50<17:51,  2.99s/it]
Traceback (most recent call last):
  File "/content/SplaTAM/scripts/splatam.py", line 1014, in <module>
    rgbd_slam(experiment.config)
  File "/content/SplaTAM/scripts/splatam.py", line 858, in rgbd_slam
    params, variables = prune_gaussians(params, variables, optimizer, iter, config['mapping']['pruning_dict'])
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/SplaTAM/utils/slam_external.py", line 180, in prune_gaussians
    params, variables = remove_points(to_remove, params, variables, optimizer)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/SplaTAM/utils/slam_external.py", line 149, in remove_points
    group["params"][0] = torch.nn.Parameter((group["params"][0][to_keep].requires_grad_(True)))
                                             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
M

In [ ]:
#Use this cell to resume splatam run from last checkpoint, if something goes wrong

def resume_from_checkpoint(seq, ckpt):
    # Restore from Drive
    src = f"/content/drive/MyDrive/EE243 Project/turtlebot_{seq}.py"
    dst = f"/content/SplaTAM/configs/turtlebot_{seq}.py"
    shutil.copy(src, dst)

    with open(dst) as f:
        content = f.read()

    # Point to local data
    content = content.replace(
        '"/content/drive/MyDrive/EE243 Project/mcap2nerfcapturedataset_results"',
        '"/content/data"'
    )
    # Resume from checkpoint
    content = content.replace('load_checkpoint=False', 'load_checkpoint=True')
    content = content.replace('checkpoint_time_idx=0', f'checkpoint_time_idx={ckpt}')

    with open(dst, "w") as f:
        f.write(content)
    print(f"{seq}: configured to resume from frame {ckpt}")

resume_from_checkpoint("seq1", 600)
resume_from_checkpoint("seq2", 500)

seq1: configured to resume from frame 600
seq2: configured to resume from frame 500


In [ ]:
# and then relaunch

for seq in ["seq1", "seq2"]:
    os.makedirs(f"/content/drive/MyDrive/EE243 Project/outputs/turtlebot_{seq}", exist_ok=True)
    proc = subprocess.Popen(
        ["python", "-u", "scripts/splatam.py", f"configs/turtlebot_{seq}.py"],
        cwd="/content/SplaTAM",
        stdout=open(f"/content/drive/MyDrive/EE243 Project/outputs/turtlebot_{seq}/run.log", "w"),
        stderr=subprocess.STDOUT
    )
    print(f"Launched {seq} PID {proc.pid}")

Launched seq1 PID 14219
Launched seq2 PID 14221


In [ ]:
# Monitor run
for seq in ["seq1", "seq2"]:
    print(f"\n\n=== {seq} ===")
    !tail -10 "/content/drive/MyDrive/EE243 Project/outputs/turtlebot_{seq}/run.log"



=== seq1 ===
Average Mapping/Frame Time: 1.833046046805469 s
Evaluating Final Parameters ...
100%|██████████| 873/873 [15:56<00:00,  1.10s/it]
Final Average ATE RMSE: 209.74 cm
Average PSNR: 20.43
Average Depth RMSE: 15.48 cm
Average Depth L1: 15.48 cm
Average MS-SSIM: 0.812
Average LPIPS: 0.220
Saving parameters to: /content/drive/MyDrive/EE243 Project/outputs/turtlebot_seq1


=== seq2 ===
Average Mapping/Frame Time: 1.9111417925104182 s
Evaluating Final Parameters ...
100%|██████████| 876/876 [14:50<00:00,  1.02s/it]
Final Average ATE RMSE: 121.26 cm
Average PSNR: 26.18
Average Depth RMSE: 4.80 cm
Average Depth L1: 4.80 cm
Average MS-SSIM: 0.906
Average LPIPS: 0.109
Saving parameters to: /content/drive/MyDrive/EE243 Project/outputs/turtlebot_seq2


#Testing on robot-mounted lab + desk footage

In [ ]:
!python "/content/drive/MyDrive/EE243 Project/mcap2nerfcapturedataset_results/mcap2dataset.py" \
    --mcap "/content/drive/MyDrive/EE243 Project/data/robot_lab_spare/robot_lab_spare_0.mcap" \
    --out  "/content/drive/MyDrive/EE243 Project/mcap2nerfcapturedataset_results/seq3" \
    --rgb-topic   /camera/camera/color/image_raw \
    --depth-topic /camera/camera/depth/image_rect_raw \
    --rgb-info    /camera/camera/color/camera_info \
    --depth-info  /camera/camera/color/camera_info

[pass 1] reading CameraInfo...
  color: 640x480 K=
[[382.26586914   0.         316.66802979]
 [  0.         381.64974976 247.90286255]
 [  0.           0.           1.        ]]
  D=[-5.62762581e-02  6.41128719e-02 -6.83844046e-05 -9.62164253e-04
 -2.00389363e-02]
  depth: 640x480 K=
[[382.26586914   0.         316.66802979]
 [  0.         381.64974976 247.90286255]
 [  0.           0.           1.        ]]
  D=[-5.62762581e-02  6.41128719e-02 -6.83844046e-05 -9.62164253e-04
 -2.00389363e-02]
  undistorted color K=
[[378.27498745   0.         315.78998898]
 [  0.         378.06932219 247.77477812]
 [  0.           0.           1.        ]]
[pass 2] loading images...
  loaded 966 rgb / 963 depth messages
[pass 3] syncing + saving pairs...
  saved frame 0 (depth valid 80.6%)
  saved frame 50 (depth valid 80.4%)
  saved frame 100 (depth valid 80.7%)
  saved frame 150 (depth valid 80.4%)
  saved frame 200 (depth valid 80.1%)
  saved frame 250 (depth valid 82.0%)
  saved frame 300 (depth v

In [ ]:
!python "/content/drive/MyDrive/EE243 Project/mcap2nerfcapturedataset_results/mcap2dataset.py" \
    --mcap "/content/drive/MyDrive/EE243 Project/data/robot_desk_seq4/robot_desk_seq4_0.mcap" \
    --out  "/content/drive/MyDrive/EE243 Project/mcap2nerfcapturedataset_results/seq4" \
    --rgb-topic   /camera/camera/color/image_raw \
    --depth-topic /camera/camera/depth/image_rect_raw \
    --rgb-info    /camera/camera/color/camera_info \
    --depth-info  /camera/camera/color/camera_info

[pass 1] reading CameraInfo...
  color: 640x480 K=
[[382.0581665    0.         316.66802979]
 [  0.         381.44238281 247.90286255]
 [  0.           0.           1.        ]]
  D=[-5.62762581e-02  6.41128719e-02 -6.83844046e-05 -9.62164253e-04
 -2.00389363e-02]
  depth: 640x480 K=
[[382.0581665    0.         316.66802979]
 [  0.         381.44238281 247.90286255]
 [  0.           0.           1.        ]]
  D=[-5.62762581e-02  6.41128719e-02 -6.83844046e-05 -9.62164253e-04
 -2.00389363e-02]
  undistorted color K=
[[378.07487586   0.         315.78951743]
 [  0.         377.86953995 247.77470342]
 [  0.           0.           1.        ]]
[pass 2] loading images...
  loaded 964 rgb / 959 depth messages
[pass 3] syncing + saving pairs...
  saved frame 0 (depth valid 72.1%)
  saved frame 50 (depth valid 75.3%)
  saved frame 100 (depth valid 75.0%)
  saved frame 150 (depth valid 81.6%)
  saved frame 200 (depth valid 78.2%)
  saved frame 250 (depth valid 79.6%)
  saved frame 300 (depth v

In [ ]:
os.makedirs("/content/data", exist_ok=True)
for seq in ["seq3", "seq4"]:
    if not os.path.exists(f"/content/data/{seq}"):
        print(f"Copying {seq}...")
        shutil.copytree(
            f"/content/drive/MyDrive/EE243 Project/mcap2nerfcapturedataset_results/{seq}",
            f"/content/data/{seq}"
        )
print("Done.")

Copying seq3...
Copying seq4...
Done.


In [ ]:
for seq in ["seq3", "seq4"]:
    os.makedirs(f"/content/drive/MyDrive/EE243 Project/outputs/turtlebot_{seq}", exist_ok=True)
    proc = subprocess.Popen(
        ["python", "-u", "scripts/splatam.py", f"configs/turtlebot_{seq}.py"],
        cwd="/content/SplaTAM",
        stdout=open(f"/content/drive/MyDrive/EE243 Project/outputs/turtlebot_{seq}/run.log", "w"),
        stderr=subprocess.STDOUT
    )
    print(f"Launched {seq} PID {proc.pid}")

Launched seq3 PID 28456
Launched seq4 PID 28458


In [ ]:
# code to resume since runtime died
resume_from_checkpoint("seq3", 700)
resume_from_checkpoint("seq4", 600)

In [ ]:
!tail -10 "/content/drive/MyDrive/EE243 Project/outputs/turtlebot_seq3/run.log"

Average Mapping/Frame Time: 2.7581729381256683 s
Evaluating Final Parameters ...
100%|██████████| 963/963 [16:24<00:00,  1.02s/it]
Final Average ATE RMSE: 85.52 cm
Average PSNR: 15.80
Average Depth RMSE: 5.07 cm
Average Depth L1: 5.07 cm
Average MS-SSIM: 0.731
Average LPIPS: 0.295
Saving parameters to: /content/drive/MyDrive/EE243 Project/outputs/turtlebot_seq3


In [ ]:
!tail -10 "/content/drive/MyDrive/EE243 Project/outputs/turtlebot_seq4/run.log"

Average Mapping/Frame Time: 2.3572192618300796 s
Evaluating Final Parameters ...
100%|██████████| 958/958 [15:54<00:00,  1.00it/s]
Final Average ATE RMSE: 26.50 cm
Average PSNR: 23.52
Average Depth RMSE: 10.50 cm
Average Depth L1: 10.50 cm
Average MS-SSIM: 0.897
Average LPIPS: 0.135
Saving parameters to: /content/drive/MyDrive/EE243 Project/outputs/turtlebot_seq4


- Used robot_lab_spare instead of robot_lab_seq3